# Phase 8: τ²-bench telecom, three arms (baseline, validator, shallow KiL)

**Goal:** test Hypothesis 1 in a domain where the **user also acts**. 40 tasks (official test split) × 8 trials = 320 dialogues per arm, same frozen settings as 11.

| Arm | What changes | Uses the graph? |
|---|---|---|
| baseline | τ² as it ships: prose policy (`main_policy.md` + `tech_support_manual.md`) | no |
| validator | + runtime validator on the agent's write tools: 5 data rules (P1, P7, S2, S3, F1) | yes (as code) |
| shallow | + the whole policy graph as text rules after the prose policy (about 1,300 words) | yes (as text) |

**Telecom policy graph** (`telecom_policy_graph.json`): same schema as the airline graph, plus `procedures` for the user-side fixes.

| Part | Count | Can the validator check it? |
|---|---|---|
| Global rules | 8 | no (dialogue) |
| Agent action conditions | 12 (7 data, 5 dialogue) | 5 of the 7 data rules; the other 2 (line ownership, "roaming not enabled yet") are already handled by the tools |
| Troubleshooting steps (3 procedures) | 17 | no: they happen on the user's phone |

*The first draft of the graph was written with AI help; every rule must be checked against the two policy files.*

**Validator:** a blocked call is returned to the agent as a rejected tool message, so τ² skips it when it replays the dialogue for scoring. The gold-check cell (cell 11) replays the gold actions of all 114 telecom base tasks (the 40 test tasks are among them) through the validator: none may be blocked.

**How to run:** set `ARM` in **cell 12**, then run cells 10 → 13. One arm per run: `baseline`, then `validator`, then `shallow`. Check cost and pass^1 after baseline before spending more.

**Do not re-run an arm that has finished.** With `--auto-resume`, τ² may remove dialogues that ended in errors and run them again, which changes the results (this happened once to baseline: 320 → 318 dialogues; fixed by re-running the 2 missing ones).

**Actual cost and time** (USD, per arm): baseline 17.28, validator 16.32, shallow 19.92 (longer prompt); about 1.5–2 hours each. Safe to stop an unfinished run: re-running cell 12 continues where it stopped.

**Runtime:** CPU. Needs Colab Secret `GEMINI_API_KEY`. **Run first:** 11 (clones and installs τ²).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# re-install tau2-bench from the clone made in 11 (a new Colab runtime forgets installs)
%cd /content/drive/MyDrive/cmsc691-10-tau2/tau2-bench
!git log -1 --format="tau2-bench commit %h"
!pip install -q -e .
print("installed")

/content/drive/MyDrive/cmsc691-10-tau2/tau2-bench
tau2-bench commit da0835f
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for tau2 (pyproject.toml) ... done
installed


In [3]:
%%writefile /content/drive/MyDrive/cmsc691-10-tau2/settings_tau2.py
# frozen settings for every tau2-bench arm (telecom stretch goal)
# kept the same as the airline Gemini arms (settings_gemini.py) wherever tau2 allows it
MODEL = "gemini/gemini-3.1-flash-lite"   # agent and user simulator, same model as 05G-08
PRICE_IN, PRICE_OUT = 0.25, 1.50         # USD per 1M tokens, Gemini price list; for cost estimates only
TEMPERATURE = 0.0                        # agent temperature (tau2 default too)
USER_TEMPERATURE = 1.0                   # user temperature: same as my airline arms (tau2 default is 0.0)
MAX_STEPS = 100                          # tau2 default is 200; telecom needs more than tau-bench's 30
SEED = 300                               # tau2 default
CONCURRENCY = 4                          # same as airline; does not change results
DOMAIN = "telecom"
SPLIT = "test"                           # official tau2 telecom test split: 40 tasks
N_TRIALS = 8
COMMIT = "da0835f6c67baef58bdda18f2a3b037dfb3a1754"   # tau2-bench version, pinned for reproducibility

Overwriting /content/drive/MyDrive/cmsc691-10-tau2/settings_tau2.py


In [31]:
%%writefile /content/drive/MyDrive/cmsc691-10-tau2/analyze_tau2.py
# read a tau2 results.json: reward, turns, tokens and cost per dialogue
import json

from settings_tau2 import PRICE_IN, PRICE_OUT


def load(path):
    return json.load(open(path))


def rows(res):
    out = []
    for s in res["simulations"]:
        tok = {"agent": [0, 0], "user": [0, 0]}
        turns = {"agent": 0, "user": 0, "agent_tool": 0, "user_tool": 0}
        for m in s.get("messages") or []:
            role = m.get("role")
            who = "agent" if role == "assistant" else ("user" if role == "user" else None)
            if who:
                turns[who] += 1
                if m.get("tool_calls"):
                    turns[f"{who}_tool"] += len(m["tool_calls"])
                u = m.get("usage") or {}
                tok[who][0] += u.get("prompt_tokens") or 0
                tok[who][1] += u.get("completion_tokens") or 0
        cost = sum(i * PRICE_IN + o * PRICE_OUT for i, o in tok.values()) / 1e6
        out.append({"task": s["task_id"], "trial": s.get("trial"),
                    "reward": (s.get("reward_info") or {}).get("reward"),
                    "end": s.get("termination_reason"),
                    "agent msgs": turns["agent"], "user msgs": turns["user"],
                    "agent tool calls": turns["agent_tool"], "user tool calls": turns["user_tool"],
                    "agent in": tok["agent"][0], "agent out": tok["agent"][1],
                    "user in": tok["user"][0], "user out": tok["user"][1],
                    "cost $": round(cost, 4)})
    return out


BLOCK_TAG = "BLOCKED by policy validator"


def blocked(res):
    """Validator blocks per dialogue (0 for arms without the validator)."""
    out = []
    for s in res["simulations"]:
        n = sum(1 for m in s.get("messages") or []
                if m.get("role") == "tool" and BLOCK_TAG in str(m.get("content")))
        out.append({"task": s["task_id"], "trial": s.get("trial"), "blocked": n})
    return out


def passk(df, max_k=8):
    """pass^k for k = 1..max_k, tau-bench formula: mean over tasks of C(c,k)/C(n,k).
    Each task uses its own number of trials n; k only goes up to the smallest n,
    so an unfinished arm can never give a value above 1."""
    from math import comb
    ok = df.assign(ok=df["reward"].fillna(0) >= 1 - 1e-6).groupby("task")["ok"].agg(["sum", "count"])
    n_min = int(ok["count"].min())
    return {k: sum(comb(int(c), k) / comb(int(n), k) for c, n in zip(ok["sum"], ok["count"])) / len(ok)
            for k in range(1, min(max_k, n_min) + 1)}, n_min


def show(df, title="", fmt=None, index=False, bar=None):
    """Display a table with a caption, clean number formats and a light style."""
    from IPython.display import display
    st = df.style
    if fmt:
        st = st.format(fmt)
    if bar:
        st = st.bar(subset=bar, color="#9ecae1", vmin=0, vmax=1)
    if not index:
        st = st.hide(axis="index")
    st = st.set_caption(title).set_table_styles([
        {"selector": "caption", "props": "caption-side: top; font-size: 15px; font-weight: bold; text-align: left; padding: 6px 0;"},
        {"selector": "th", "props": "background-color: #f2f2f2; text-align: center; padding: 4px 10px;"},
        {"selector": "td", "props": "text-align: center; padding: 4px 10px;"},
    ])
    display(st)

Overwriting /content/drive/MyDrive/cmsc691-10-tau2/analyze_tau2.py


In [6]:
%%writefile /content/drive/MyDrive/cmsc691-10-tau2/graph_text_tau2.py
# Turn the telecom policy graph (telecom_policy_graph.json) into plain text rules
# for the shallow arm: the whole text is added to the agent's system prompt,
# after the prose policy. Same content as the graph: nothing added, nothing dropped.
# Same layout as the airline graph_text.py (07), plus the user-side procedures.
import json
import os

_HERE = os.path.dirname(os.path.abspath(__file__))
GRAPH_PATH = os.environ.get("POLICY_GRAPH", os.path.join(_HERE, "telecom_policy_graph.json"))


def load(path=GRAPH_PATH):
    return json.load(open(path))


def _where(c):
    if c.get("check") == "data":
        return "check in the tool results"
    return "check in the conversation" if c.get("actor") == "user" else "your own behaviour"


def _if_fails(c):
    f = c.get("if_fails")
    if f == "transfer_to_human_agents":
        return " If not: transfer the user to a human agent."
    return ""


def action_text(g, name):
    a = next(x for x in g["actions"] if x["action"] == name)
    lines = [f"### {a['action']} (tool: {a['tool']}; from {', '.join(a['read'])})"]
    if a.get("needs_info"):
        lines.append("Ask / find first: " + "; ".join(f"{i['info']} [{i['source']}]" for i in a["needs_info"]))
    lines.append("Allowed only if ALL of these hold:")
    for k, c in enumerate(a["conditions"], 1):
        lines.append(f"  {k}. {c['condition']} [{c['source']}; {_where(c)}]{_if_fails(c)}")
    for f in a.get("facts", []):
        lines.append(f"Fact: {f['text']} [{f['source']}]")
    for f in a.get("after", []):
        lines.append(f"After the call: {f['text']} [{f['source']}]")
    return "\n".join(lines)


def procedure_text(p):
    lines = [f"### Issue: {p['issue']}",
             f"Diagnose with (user tool): {p['diagnose']}"]
    if p.get("prerequisite"):
        lines.append(f"Prerequisite: {p['prerequisite']}")
    lines.append("Check each cause; for every cause that holds, do the fix:")
    for k, s in enumerate(p["steps"], 1):
        then = f", then guide the user to {s['then']}" if s.get("then") else ""
        lines.append(f"  {k}. IF {s['if']} -> {s['fix']}{then} [{s['source']}]")
    return "\n".join(lines)


def full_text(g):
    lines = ["# Policy checklist (the same rules as the policy above, written as a graph)",
             "", "## Global rules (every conversation)"]
    for r in g["global_rules"]:
        lines.append(f"- {r['condition']} [{r['source']}].")
    lines += ["", "## Agent actions"]
    for a in g["actions"]:
        lines += ["", action_text(g, a["action"])]
    lines += ["", "## Troubleshooting procedures (the user does these on their own phone; guide them step by step)",
              "Order: cellular service first, then mobile data, then MMS."]
    for p in g["procedures"]:
        lines += ["", procedure_text(p)]
    return "\n".join(lines)


if __name__ == "__main__":
    t = full_text(load())
    print(t)
    print(f"\n[{len(t.split())} words]")

Overwriting /content/drive/MyDrive/cmsc691-10-tau2/graph_text_tau2.py


In [7]:
%%writefile /content/drive/MyDrive/cmsc691-10-tau2/validator_tau2.py
# Runtime validator for tau2-bench telecom (the control arm, like 06G for airline).
# Only the agent-side DATA rules of the telecom policy graph can be checked in code:
#   P1  send_payment_request: the bill status is Overdue
#   P7  send_payment_request: no other bill of the customer is Awaiting Payment
#   S2  resume_line: the customer has no Overdue / Awaiting Payment bill left
#   S3  resume_line: the line's contract end date is not in the past
#   F1  refuel_data: at most 2 GB
# The user-side fixes (airplane mode, APN, ...) happen on the user's phone and the
# dialogue rules (confirm price, reboot after resume, ...) are only in the conversation:
# a validator cannot see them.
# A blocked call is returned to the agent as a REJECTED tool message (error=True,
# execution_status="rejected"), so tau2 skips it when it replays the dialogue for scoring.
# The gold run in the evaluator calls tools directly (make_tool_call), so the
# validator never touches the answer key.
from datetime import date, datetime

from tau2.data_model.message import ToolMessage
from tau2.environment.environment import Environment

TAG = "BLOCKED by policy validator"
CHECKED = {"send_payment_request", "resume_line", "refuel_data"}
_original = Environment.get_response


def _status(x):
    return getattr(x, "value", x)


def violations(tools, name, args):
    """List of (rule, reason) the call breaks; [] if fine or if it cannot be checked."""
    out = []
    try:
        if name == "send_payment_request":
            cust = tools.get_customer_by_id(args["customer_id"])
            bill = tools._get_bill_by_id(args["bill_id"])
            if _status(bill.status) != "Overdue":
                out.append(("P1", f"bill {bill.bill_id} is {_status(bill.status)}, not Overdue"))
            waiting = [b for b in cust.bill_ids
                       if b != args["bill_id"] and _status(tools._get_bill_by_id(b).status) == "Awaiting Payment"]
            if waiting:
                out.append(("P7", f"bill {waiting[0]} is already Awaiting Payment"))
        elif name == "resume_line":
            cust = tools.get_customer_by_id(args["customer_id"])
            unpaid = [b for b in cust.bill_ids
                      if _status(tools._get_bill_by_id(b).status) in ("Overdue", "Awaiting Payment")]
            if unpaid:
                out.append(("S2", f"overdue bill(s) not paid yet: {', '.join(unpaid)}"))
            line = tools._get_line_by_id(args["line_id"])
            end = line.contract_end_date
            if isinstance(end, str):
                end = date.fromisoformat(end[:10])
            if isinstance(end, datetime):
                end = end.date()
            if end is not None and end < tools._get_today():
                out.append(("S3", f"contract end date {end} is in the past"))
        elif name == "refuel_data":
            if float(args.get("gb_amount", 0)) > 2:
                out.append(("F1", f"{args['gb_amount']} GB is more than the 2 GB maximum"))
    except Exception:
        return []          # bad ids etc.: let the tool itself return its normal error
    return out


def get_response(self, tool_call):
    if tool_call.requestor == "assistant" and tool_call.name in CHECKED:
        v = violations(self.tools, tool_call.name, tool_call.arguments or {})
        if v:
            reason = "; ".join(f"[{r}] {why}" for r, why in v)
            return ToolMessage(id=tool_call.id, role="tool", requestor=tool_call.requestor,
                               content=f"{TAG}: {reason}. The call was not executed.",
                               error=True, execution_status="rejected")
    return _original(self, tool_call)


def install():
    Environment.get_response = get_response

Overwriting /content/drive/MyDrive/cmsc691-10-tau2/validator_tau2.py


In [8]:
%%writefile /content/drive/MyDrive/cmsc691-10-tau2/run_arm.py
# Run one telecom arm with the normal tau2 CLI, after patching it for that arm.
#   python run_arm.py baseline  run --domain telecom ...
#   python run_arm.py validator run --domain telecom ...
#   python run_arm.py shallow   run --domain telecom ...
# baseline : tau2 as it ships (prose policy)
# validator: + runtime validator on the agent's write tools (validator_tau2.py)
# shallow  : + the policy graph as text rules after the prose policy (graph_text_tau2.py)
# Everything else (model, temperatures, tasks, seed) is the same command line.
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
ARM = sys.argv.pop(1)
assert ARM in ("baseline", "validator", "shallow"), ARM

from tau2.registry import registry  # noqa: E402

if ARM == "validator":
    import validator_tau2
    validator_tau2.install()
elif ARM == "shallow":
    import graph_text_tau2
    GRAPH = "\n<policy_graph>\n" + graph_text_tau2.full_text(graph_text_tau2.load()) + "\n</policy_graph>"
    _make = registry._domains["telecom"]

    def make_env(*args, **kwargs):
        env = _make(*args, **kwargs)
        env.policy = env.policy + GRAPH     # prose policy kept, graph text added after it
        return env

    registry._domains["telecom"] = make_env

print(f"[run_arm] arm = {ARM}", flush=True)
from tau2.cli import main  # noqa: E402
sys.argv[0] = "tau2"
main()

Overwriting /content/drive/MyDrive/cmsc691-10-tau2/run_arm.py


In [9]:
%%writefile /content/drive/MyDrive/cmsc691-10-tau2/telecom_policy_graph.json
{
  "how_to_fill": "Policy graph for tau2-bench telecom, same schema as airline_policy_graph.json. Each condition: condition, actor (agent or user), check (data = the agent can check it in tool results; dialogue = only visible in the conversation; device = only visible on the user's phone), source (bullet id), if_fails (default deny, G3). any_of / all_of = OR / AND. 'order' in needs_info = must be asked in that order. 'must_happen_before' = this step must be done before the named action. New for telecom: 'procedures' = troubleshooting steps the agent must guide the USER to do on their own phone (diagnose with a user read tool, then fix with a user write tool), with 'prerequisite' and 'then' for order.",
  "status": "First draft written with AI help from main_policy.md and tech_support_manual.md (tau2-bench commit da0835f); every rule must be checked by hand against the two files.",
  "rule_of_thumb": "Graph = same content as main_policy.md + tech_support_manual.md, only the form differs. Nothing added, nothing dropped.",
  "source_ids": "Main policy: G1-G6 global, L1 customer lookup, P1-P7 overdue bill payment, S1-S4 line suspension, F1-F4 data refueling, CP1 change plan, RO1 data roaming. Tech support manual: T0 identify customer + try all ways before transfer, SV1-SV4 cellular service, MD1-MD7 mobile data, MM1-MM5 MMS. Numbered top to bottom in each file.",
  "info_only": [
    {
      "source": "D1",
      "text": "customer profile fields: customer ID, full name, date of birth, email, phone number, address, account status, created date, payment methods, line IDs, bill IDs, last extension date, goodwill credit usage"
    },
    {
      "source": "D2",
      "text": "account status: Active, Suspended, Pending Verification, Closed; line status: Active, Suspended, Pending Activation, Closed; bill status: Draft, Issued, Paid, Overdue, Awaiting Payment, Disputed"
    },
    {
      "source": "D3",
      "text": "the current time is 2025-02-25 12:08:00 EST"
    }
  ],
  "global_rules": [
    {
      "condition": "give no information, knowledge or procedures not given by the user or available tools; no subjective recommendations or comments",
      "actor": "agent",
      "check": "dialogue",
      "source": "G1",
      "verified": false
    },
    {
      "condition": "make one tool call at a time; never call a tool and reply to the user in the same turn",
      "actor": "agent",
      "check": "dialogue",
      "source": "G2",
      "verified": false
    },
    {
      "condition": "deny user requests that are against this policy",
      "actor": "agent",
      "check": "dialogue",
      "source": "G3",
      "verified": false
    },
    {
      "condition": "transfer to a human agent if and only if the request cannot be handled within the scope of the agent's actions",
      "actor": "agent",
      "check": "dialogue",
      "source": "G4",
      "verified": false
    },
    {
      "condition": "to transfer: first call transfer_to_human_agents, then send the message 'YOU ARE BEING TRANSFERRED TO A HUMAN AGENT. PLEASE HOLD ON.'",
      "actor": "agent",
      "check": "dialogue",
      "source": "G5",
      "verified": false
    },
    {
      "condition": "try your best (all possible ways) to resolve the issue before transferring to a human agent",
      "actor": "agent",
      "check": "dialogue",
      "source": "G6",
      "verified": false
    },
    {
      "condition": "look up the customer by phone number, customer ID, or full name with date of birth (date of birth is required for a name lookup)",
      "actor": "agent",
      "check": "data",
      "source": "L1",
      "verified": false
    },
    {
      "condition": "for technical support, first identify the customer",
      "actor": "agent",
      "check": "data",
      "source": "T0",
      "verified": false
    }
  ],
  "actions": [
    {
      "action": "send_payment_request",
      "tool": "send_payment_request",
      "read": [
        "P1",
        "P2",
        "P3",
        "P4",
        "P7"
      ],
      "needs_info": [
        {
          "info": "customer (identified by phone number, customer ID, or full name + date of birth)",
          "source": "L1",
          "verified": false,
          "order": 1
        },
        {
          "info": "the overdue bill",
          "source": "P1",
          "verified": false
        }
      ],
      "conditions": [
        {
          "condition": "the bill status is Overdue (check the bill before sending)",
          "actor": "agent",
          "check": "data",
          "source": "P1",
          "verified": false
        },
        {
          "condition": "the agent has checked the bill amount due",
          "actor": "agent",
          "check": "dialogue",
          "source": "P2",
          "verified": false
        },
        {
          "condition": "the customer has no other bill in AWAITING PAYMENT status (only one at a time)",
          "actor": "agent",
          "check": "data",
          "source": "P7",
          "verified": false
        }
      ],
      "facts": [
        {
          "text": "sending the request changes the bill status to AWAITING PAYMENT",
          "source": "P3",
          "verified": false
        },
        {
          "text": "the send payment request tool does not check if the bill is overdue; the agent must check it before the call",
          "source": "P7",
          "verified": false
        }
      ],
      "after": [
        {
          "text": "tell the user a payment request was sent and that they should check it with check_payment_request (user tool)",
          "source": "P4",
          "verified": false
        },
        {
          "text": "if the user accepts, the payment is made with make_payment (user tool); the bill then becomes PAID",
          "source": "P5",
          "verified": false
        },
        {
          "text": "check that the bill status is PAID before telling the user the bill is paid",
          "source": "P6",
          "verified": false
        }
      ]
    },
    {
      "action": "resume_line",
      "tool": "resume_line",
      "read": [
        "S1",
        "S2",
        "S3",
        "S4",
        "SV4"
      ],
      "needs_info": [
        {
          "info": "customer and the line (phone number) the user means",
          "source": "L1",
          "verified": false,
          "order": 1
        }
      ],
      "conditions": [
        {
          "condition": "the user has paid all their overdue bills",
          "actor": "agent",
          "check": "data",
          "source": "S2",
          "verified": false
        },
        {
          "condition": "the line's contract end date is not in the past (even if all bills are paid)",
          "actor": "agent",
          "check": "data",
          "source": "S3",
          "verified": false,
          "if_fails": "transfer_to_human_agents"
        }
      ],
      "facts": [
        {
          "text": "a line can be suspended because the user has an overdue bill or because the line's contract end date is in the past; a suspended line has no service",
          "source": "S1",
          "verified": false
        },
        {
          "text": "if the suspension cannot be lifted by the agent, escalate to technical support",
          "source": "SV4",
          "verified": false
        }
      ],
      "after": [
        {
          "text": "after the line is resumed, the user must reboot the device (reboot_device) to get service; then verify service is restored",
          "source": "S4",
          "verified": false
        }
      ]
    },
    {
      "action": "refuel_data",
      "tool": "refuel_data",
      "read": [
        "F1",
        "F2",
        "F3",
        "F4",
        "MD6"
      ],
      "needs_info": [
        {
          "info": "how much data the user wants to refuel",
          "source": "F2",
          "verified": false,
          "order": 1
        },
        {
          "info": "the line of the phone number the user provided",
          "source": "F4",
          "verified": false
        }
      ],
      "conditions": [
        {
          "condition": "at most 2 GB is refueled",
          "actor": "agent",
          "check": "data",
          "source": "F1",
          "verified": false
        },
        {
          "condition": "the agent confirmed the price (plan's refueling price per GB) with the user before the call",
          "actor": "user",
          "check": "dialogue",
          "source": "F3",
          "verified": false
        },
        {
          "condition": "the data is applied to the line associated with the phone number the user provided",
          "actor": "agent",
          "check": "data",
          "source": "F4",
          "verified": false
        }
      ],
      "facts": [
        {
          "text": "if data usage exceeds the plan's data limit, data connectivity is lost; the user can change to a plan with more data or refuel data",
          "source": "MD6",
          "verified": false
        }
      ]
    },
    {
      "action": "change_plan",
      "tool": "(no agent tool for this in the telecom tool list)",
      "read": [
        "CP1"
      ],
      "needs_info": [
        {
          "info": "which line the user wants to change the plan for",
          "source": "CP1",
          "verified": false,
          "order": 1
        },
        {
          "info": "the plan the user selects from the available plans",
          "source": "CP1",
          "verified": false,
          "order": 2
        }
      ],
      "conditions": [
        {
          "condition": "the agent gathered the available plans, calculated the price of the new plan and confirmed the price with the user before applying it",
          "actor": "user",
          "check": "dialogue",
          "source": "CP1",
          "verified": false
        }
      ],
      "facts": []
    },
    {
      "action": "enable_roaming",
      "tool": "enable_roaming",
      "read": [
        "RO1",
        "MD3"
      ],
      "needs_info": [
        {
          "info": "the line of the phone number the user provided",
          "source": "MD3",
          "verified": false
        }
      ],
      "conditions": [
        {
          "condition": "the user is traveling outside their home network (abroad)",
          "actor": "user",
          "check": "dialogue",
          "source": "RO1",
          "verified": false
        },
        {
          "condition": "the line is not roaming enabled yet",
          "actor": "agent",
          "check": "data",
          "source": "RO1",
          "verified": false
        }
      ],
      "facts": [
        {
          "text": "roaming is enabled at no cost for the user",
          "source": "RO1",
          "verified": false
        },
        {
          "text": "the user must also turn Data Roaming ON on the phone (toggle_roaming)",
          "source": "MD3",
          "verified": false
        }
      ]
    },
    {
      "action": "transfer_to_human_agents",
      "tool": "transfer_to_human_agents",
      "read": [
        "G4",
        "G5",
        "G6",
        "SV2",
        "SV4"
      ],
      "needs_info": [],
      "conditions": [
        {
          "condition": "the request cannot be handled within the scope of the agent's actions, after trying all possible ways",
          "actor": "agent",
          "check": "dialogue",
          "source": "G4",
          "verified": false
        }
      ],
      "facts": [
        {
          "text": "escalate (transfer) when the SIM is Locked (PIN/PUK), or when a suspension cannot be lifted by the agent",
          "source": "SV2",
          "verified": false
        }
      ]
    }
  ],
  "procedures": [
    {
      "issue": "no cellular service",
      "diagnose": "check_status_bar (no signal strength shown = no service)",
      "prerequisite": null,
      "steps": [
        {
          "if": "airplane mode is ON (check_status_bar or check_network_status)",
          "fix": "guide the user to toggle_airplane_mode to turn it OFF",
          "source": "SV1",
          "verified": false
        },
        {
          "if": "check_sim_status shows Missing",
          "fix": "guide the user to reseat_sim_card",
          "source": "SV2",
          "verified": false
        },
        {
          "if": "check_sim_status shows Locked (PIN/PUK)",
          "fix": "escalate to technical support (transfer)",
          "source": "SV2",
          "verified": false
        },
        {
          "if": "check_apn_settings shows Incorrect",
          "fix": "guide the user to reset_apn_settings",
          "then": "reboot_device",
          "source": "SV3",
          "verified": false
        },
        {
          "if": "the line is suspended",
          "fix": "follow resume_line rules; if allowed, resume and verify service; if not, escalate",
          "then": "reboot_device",
          "source": "SV4",
          "verified": false
        }
      ]
    },
    {
      "issue": "mobile data not working or slow",
      "diagnose": "run_speed_test ('no connection' = no data; any speed below Excellent = slow)",
      "prerequisite": "cellular service must work first",
      "steps": [
        {
          "if": "airplane mode is ON",
          "fix": "guide the user to toggle_airplane_mode to turn it OFF",
          "source": "MD1",
          "verified": false
        },
        {
          "if": "check_network_status shows mobile data disabled",
          "fix": "guide the user to toggle_data to turn it ON",
          "source": "MD2",
          "verified": false
        },
        {
          "if": "the user is abroad and Data Roaming is OFF on the phone",
          "fix": "guide the user to toggle_roaming to turn it ON",
          "source": "MD3",
          "verified": false
        },
        {
          "if": "the user is abroad and the line is not roaming enabled",
          "fix": "enable_roaming (agent tool, no cost)",
          "source": "MD3",
          "verified": false
        },
        {
          "if": "check_data_restriction_status shows Data Saver mode is ON",
          "fix": "guide the user to toggle_data_saver_mode to turn it OFF",
          "source": "MD4",
          "verified": false
        },
        {
          "if": "check_vpn_status shows VPN ON and connected with performance Poor",
          "fix": "guide the user to disconnect_vpn",
          "source": "MD5",
          "verified": false
        },
        {
          "if": "data usage of the line exceeds the plan's data limit",
          "fix": "offer a plan with more data or refuel data (refuel_data rules)",
          "source": "MD6",
          "verified": false
        },
        {
          "if": "check_network_mode_preference shows 2G or 3G",
          "fix": "guide the user to set_network_mode_preference with mode '4g_5g_preferred'",
          "source": "MD7",
          "verified": false
        }
      ]
    },
    {
      "issue": "cannot send MMS (picture/video messages)",
      "diagnose": "can_send_mms",
      "prerequisite": "cellular service and mobile data (any speed) must work first",
      "steps": [
        {
          "if": "check_network_status shows 2G",
          "fix": "guide the user to set_network_mode_preference to a mode with 3G/4G/5G (e.g. '4g_5g_preferred' or '4g_only')",
          "source": "MM2",
          "verified": false
        },
        {
          "if": "check_apn_settings shows the MMSC URL is not set",
          "fix": "guide the user to reset_apn_settings",
          "then": "reboot_device",
          "source": "MM3",
          "verified": false
        },
        {
          "if": "check_wifi_calling_status shows Wi-Fi Calling is ON",
          "fix": "guide the user to toggle_wifi_calling to turn it OFF",
          "source": "MM4",
          "verified": false
        },
        {
          "if": "check_app_permissions(app_name='messaging') does not list 'storage' and 'sms' as granted",
          "fix": "guide the user to grant_app_permission(app_name='messaging', permission='storage') and (permission='sms')",
          "source": "MM5",
          "verified": false
        }
      ]
    }
  ]
}

Overwriting /content/drive/MyDrive/cmsc691-10-tau2/telecom_policy_graph.json


In [33]:
# API key (Colab Secrets), data folder, imports
import os, sys, json, importlib, shlex
import pandas as pd
from IPython.display import display
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["TAU2_DATA_DIR"] = "/content/drive/MyDrive/cmsc691-10-tau2/tau2-bench/data"
sys.path.insert(0, "/content/drive/MyDrive/cmsc691-10-tau2")
sys.path.insert(0, "/content/drive/MyDrive/cmsc691-10-tau2/tau2-bench/src")   # the editable install is only seen after a restart; this makes tau2 importable now
import settings_tau2 as S, analyze_tau2 as A, graph_text_tau2 as GT
for m in (S, A, GT): importlib.reload(m)
g = GT.load()
conds = [c for a in g["actions"] for c in a["conditions"]]
A.show(pd.DataFrame([
    ("global rules", len(g["global_rules"])),
    ("agent actions", len(g["actions"])),
    ("  conditions (data / dialogue)", f'{len(conds)} ({sum(c["check"] == "data" for c in conds)} / {sum(c["check"] == "dialogue" for c in conds)})'),
    ("troubleshooting procedures (user side)", len(g["procedures"])),
    ("  steps", sum(len(p["steps"]) for p in g["procedures"])),
    ("rules the validator enforces", "5 (P1, P7, S2, S3, F1)"),
    ("shallow text added to the prompt", f'{len(GT.full_text(g).split())} words'),
], columns=["telecom policy graph", "count"]), "Policy graph")
A.show(pd.DataFrame([(k, getattr(S, k)) for k in ["MODEL", "TEMPERATURE", "USER_TEMPERATURE", "MAX_STEPS", "SEED", "CONCURRENCY", "DOMAIN", "SPLIT", "N_TRIALS"]],
                    columns=["setting", "value"]).astype(str), "Frozen settings (settings_tau2.py)")

telecom policy graph,count
global rules,8
agent actions,6
conditions (data / dialogue),12 (7 / 5)
troubleshooting procedures (user side),3
steps,17
rules the validator enforces,"5 (P1, P7, S2, S3, F1)"
shallow text added to the prompt,1313 words


setting,value
MODEL,gemini/gemini-3.1-flash-lite
TEMPERATURE,0.0
USER_TEMPERATURE,1.0
MAX_STEPS,100
SEED,300
CONCURRENCY,4
DOMAIN,telecom
SPLIT,test
N_TRIALS,8


In [34]:
# free check (no API): replay the gold actions of all 114 telecom tasks through the validator.
# It must block none of them, or the validator would fight the answer key (the airline problem).
import validator_tau2 as V
from tau2.registry import registry
from tau2.domains.telecom.environment import get_tasks
from tau2.data_model.message import ToolCall
import logging; logging.disable(logging.WARNING)
from loguru import logger; logger.remove()
_saved = V.Environment.get_response
V.install()
make = registry.get_env_constructor("telecom")
checked, blocked = 0, []
for t in get_tasks():
    env = make()
    env.set_state(initialization_data=t.initial_state.initialization_data,
                  initialization_actions=t.initial_state.initialization_actions, message_history=[])
    for i, a in enumerate(t.evaluation_criteria.actions or []):
        checked += a.requestor == "assistant" and a.name in V.CHECKED
        r = env.get_response(ToolCall(id=f"g{i}", name=a.name, arguments=a.arguments, requestor=a.requestor))
        if V.TAG in r.content:
            blocked.append((t.id, a.name, r.content))
V.Environment.get_response = _saved      # undo: this notebook only patches inside run_arm.py
A.show(pd.DataFrame([("telecom tasks (base)", len(get_tasks())), ("gold calls on checked tools", checked),
                     ("blocked by the validator", len(blocked))], columns=["gold check", "count"]),
       "Validator vs answer key (must be 0 blocked)")
if blocked:
    A.show(pd.DataFrame(blocked, columns=["task", "tool", "message"]), "Blocked gold calls")

gold check,count
telecom tasks (base),114
gold calls on checked tools,62
blocked by the validator,0


In [35]:
# choose the arm, then run: 40 tasks x 8 trials. Safe to re-run (continues where it stopped).
ARM = "shallow"         # "baseline", then "validator", then "shallow"
cmd = (f"python /content/drive/MyDrive/cmsc691-10-tau2/run_arm.py {ARM} run --domain {S.DOMAIN} --task-split-name {S.SPLIT} "
       f"--num-trials {S.N_TRIALS} --agent-llm {S.MODEL} --user-llm {S.MODEL} "
       f"--agent-llm-args {shlex.quote(json.dumps({'temperature': S.TEMPERATURE}))} "
       f"--user-llm-args {shlex.quote(json.dumps({'temperature': S.USER_TEMPERATURE}))} "
       f"--max-steps {S.MAX_STEPS} --seed {S.SEED} --max-concurrency {S.CONCURRENCY} "
       f"--communicate-scorer substring --save-to telecom_{ARM} --auto-resume")
A.show(pd.DataFrame([("arm", ARM), ("tasks x trials", f"40 x {S.N_TRIALS}"), ("save to", f"telecom_{ARM}")],
                    columns=["run", "value"]), "This run")
%cd /content/drive/MyDrive/cmsc691-10-tau2/tau2-bench
!{cmd} 2>&1 | grep -v -E "DEBUG|INFO" | tail -60

run,value
arm,shallow
tasks x trials,40 x 8
save to,telecom_shallow


/content/drive/MyDrive/cmsc691-10-tau2/tau2-bench
Skipping task 
[mms_issue]airplane_mode_on|bad_network_preference|bad_wifi_calling|break_apn_mm
s_setting|break_app_both_permissions|unseat_sim_card|user_abroad_roaming_enabled
_off[PERSONA:Hard], trial 8 because it has already been run.
Skipping task 
[mms_issue]bad_network_preference|bad_wifi_calling|break_app_sms_permission|data
_mode_off|data_usage_exceeded|unseat_sim_card|user_abroad_roaming_enabled_off[PE
RSONA:Easy], trial 8 because it has already been run.
Skipping task 
[mms_issue]airplane_mode_on|bad_network_preference|bad_wifi_calling|break_apn_mm
s_setting|break_app_sms_permission|data_mode_off|data_usage_exceeded|unseat_sim_
card|user_abroad_roaming_disabled_on[PERSONA:None], trial 8 because it has 
already been run.
Skipping task 
[mms_issue]airplane_mode_on|bad_network_preference|bad_wifi_calling|break_apn_mm
s_setting|break_app_storage_permission|data_mode_off|data_usage_exceeded|unseat_
sim_card|user_abroad_roaming_disa

In [36]:
# results of every arm that has run so far: pass^k, cost, length, validator blocks
SIM = "/content/drive/MyDrive/cmsc691-10-tau2/tau2-bench/data/simulations"
rows, curves = [], {}
for arm in ["baseline", "validator", "shallow"]:
    path = f"{SIM}/telecom_{arm}/results.json"
    if not os.path.exists(path):
        continue
    res = A.load(path)
    df = pd.DataFrame(A.rows(res))
    bl = pd.DataFrame(A.blocked(res))
    pk, n = A.passk(df)
    curves[arm] = pk
    rows.append({"arm": arm, "dialogues": len(df), "trials / task": n,
                 "pass^1": pk[1], "pass^8": pk.get(8),
                 "drop 1→8": 1 - pk[8] / pk[1] if pk.get(8) is not None and pk[1] else None,
                 "cost $": df["cost $"].sum(), "$ / dialogue": df["cost $"].mean(),
                 "agent msgs": df["agent msgs"].mean(), "user tool calls": df["user tool calls"].mean(),
                 "blocked calls": int(bl["blocked"].sum()),
                 "ends": ", ".join(f"{k} {v}" for k, v in df["end"].value_counts().items())})
if rows:
    A.show(pd.DataFrame(rows), "Telecom arms (40 tasks x 8 trials each)",
           fmt={"pass^1": "{:.3f}", "pass^8": "{:.3f}", "drop 1→8": "{:.0%}", "cost $": "${:.2f}",
                "$ / dialogue": "${:.4f}", "agent msgs": "{:.1f}", "user tool calls": "{:.1f}"},
           bar=["pass^1", "pass^8"])
    curve = pd.DataFrame(curves).rename_axis("k").reset_index()
    A.show(curve, "pass^k curve", fmt={a: "{:.3f}" for a in curves}, bar=list(curves))
else:
    print("no arm has finished yet")

arm,dialogues,trials / task,pass^1,pass^8,drop 1→8,cost $,$ / dialogue,agent msgs,user tool calls,blocked calls,ends
baseline,320,8,0.312,0.175,44%,$17.28,$0.0540,20.1,2.5,0,"user_stop 286, max_steps 25, too_many_errors 9"
validator,320,8,0.316,0.125,60%,$16.32,$0.0510,19.1,2.4,45,"user_stop 288, max_steps 20, too_many_errors 12"
shallow,320,8,0.434,0.175,60%,$19.92,$0.0622,18.7,4.5,0,"user_stop 304, max_steps 10, too_many_errors 6"


k,baseline,validator,shallow
1,0.312,0.316,0.434
2,0.255,0.260,0.334
3,0.233,0.230,0.289
4,0.216,0.206,0.259
5,0.202,0.184,0.235
6,0.190,0.163,0.213
7,0.181,0.144,0.194
8,0.175,0.125,0.175


## Summary

**Results (40 tasks × 8 trials per arm)**

| Arm | pass^1 | pass^8 | Drop 1→8 | User tool calls / dialogue | Blocked calls | Cost (USD) |
|---|---|---|---|---|---|---|
| baseline | 0.312 | 0.175 | 44% | 2.5 | 0 | 17.28 |
| validator | 0.316 | 0.125 | 60% | 2.4 | 45 | 16.32 |
| shallow | **0.434** | 0.175 | 60% | **4.5** | 0 | 19.92 |
| **Total** | | | | | | **53.52** |

**How dialogues ended**

| Arm | User stop | Max steps | Too many errors |
|---|---|---|---|
| baseline | 286 | 25 | 9 |
| validator | 288 | 20 | 12 |
| shallow | 304 | 10 | 6 |

**Gold check (validator vs answer key)**

| Tasks | Gold calls | On checked tools | Blocked |
|---|---|---|---|
| base (114) | 516 | 62 | 0 |
| test (40) | 162 | 18 (11%) | 0 |

**Key points (significance tested in 13)**
- **Shallow beats the validator:** pass^1 is +0.118 higher, and it is at or above the validator at every k.
- **Why:** with the graph, the agent guides the user through almost 2× more phone checks and fixes (4.5 vs 2.4). A validator cannot see these.
- **But not flatter:** shallow's pass^8 falls back to the baseline's 0.175.
- **Validator barely helps:** pass^1 +0.004 over baseline, and pass^8 is lower (0.125 vs 0.175). It can check only 11% of the gold calls; the rest happen on the user's phone.
- **Note:** 2 baseline dialogues that ended in errors were re-run by τ²'s `--auto-resume` (first run pass^1 0.306).

**Next:** 13, paired Δ with bootstrap CI (as in 10), pass^1 by issue type (MMS / service / mobile data), and a hand-check of failed dialogues.